# Laboratorium 2, Część I: BERT, PEFT i GPT-2 (notatnik do nauki)

Praktyczne wprowadzenie do czterech rzeczy, które wymaga Część I zadania:

| Część | Temat | Gdzie |
|---|---|---|
| A | `BertModel`, `BertForMaskedLM`, `BertForSequenceClassification`, `BertForTokenClassification` | [Część A](#Część-A) |
| B | Biblioteka PEFT (LoRA), quicktour i przykłady z repozytorium | [Część B](#Część-B) |
| C | Powtórzenie tutoriala z Medium (PEFT-BERT: objawy -> diagnoza) | [Część C](#Część-C) |
| D | `GPT2Model`, `GPT2LMHeadModel`, `GPT2ForSequenceClassification`, `GPT2ForTokenClassification` | [Część D](#Część-D) |

## Jak korzystać z notatnika

Samo czytanie wyników to jeszcze nie rozumienie. W kluczowych miejscach są komórki **Przewidź**: wpisz swoją odpowiedź do zmiennej (np. `pred = (2, 9, 768)`) i *dopiero potem* uruchom następną komórkę, która porówna ją z prawdziwą wartością. Zostawienie `None` po prostu pokaże prawidłową odpowiedź, więc notatnik zawsze da się uruchomić od góry do dołu.

Są też krótkie ćwiczenia **Twoja kolej** z komentarzem `# TODO`. Każde ma komórkę sprawdzającą, czy Twoje rozwiązanie jest dobre.

**Jeśli coś jest niejasne, zacznij od sekcji "Podstawy w pigułce" poniżej.** Tłumaczy ona słowa, które wracają w całym notatniku.

Oznaczenia kształtów: `B` = ile zdań naraz (batch), `T` = ile tokenów ma zdanie, `H` = rozmiar wektora (768), `V` = rozmiar słownika, `L` = liczba etykiet (klas).

# Podstawy w pigułce

Zanim wejdziemy w kod, kilka słów, które będą wracać. Wszystko po ludzku, bez matematyki.

**Token.** Model nie czyta liter ani całych słów, tylko "klocki" tekstu zwane tokenami. Zwykle to słowo albo jego kawałek (np. `unhappiness` -> `un`, `##happi`, `##ness`). Każdy token ma swój numer w słowniku (`input_ids`).

**Wektor (embedding).** Każdy token jest zamieniany na listę 768 liczb. Można je traktować jak "współrzędne znaczenia": słowa o podobnym znaczeniu mają podobne wektory.

**Stan ukryty (hidden state).** To wektor tokenu **po przejściu przez model**. Na wejściu słowo "zamek" ma zawsze ten sam wektor. Po przejściu przez model jego stan ukryty zależy od kontekstu: inny w "zamek w drzwiach", inny w "zamek w Malborku". Model "rozumie", o co chodzi.

**Uwaga (attention).** Mechanizm, w którym każdy token zerka na inne tokeny i bierze od nich informacje potrzebne do zrozumienia kontekstu. Maska uwagi (`attention_mask`) mówi, które tokeny to padding i trzeba je ignorować.

**Padding.** Żeby przetwarzać kilka zdań naraz (w paczce, *batch*), muszą mieć tę samą długość. Krótsze dopychamy "wypełniaczami" (`[PAD]`).

**Kształt tensora.** `(2, 9, 768)` czytaj: "2 zdania x 9 tokenów x 768 liczb na token". Zwykła tabela, tylko trójwymiarowa.

**Logity -> prawdopodobieństwa.** Na końcu model wypluwa dla każdej możliwej odpowiedzi zwykłą liczbę (*logit*), im większa, tym bardziej model "wierzy" w tę odpowiedź. Funkcja `softmax` zamienia te liczby na prawdopodobieństwa (sumują się do 1).

**Strata (loss).** Jedna liczba mówiąca, jak bardzo model się myli (0 = idealnie). Trening polega na takim zmienianiu parametrów, żeby strata malała.

**Parametry (wagi).** Liczby w modelu, które "wyuczył się" podczas treningu. BERT-base ma ich ok. 110 milionów.

**Ciało i głowica (body i head).** Najważniejszy obraz do zapamiętania:
- **ciało** (np. `BertModel`) = ktoś, kto przeczytał pół internetu i *rozumie język*,
- **głowica** = mały dodatkowy kawałek na końcu, który zamienia to "rozumienie" na konkretną odpowiedź (etykieta, brakujące słowo itd.).

**Pretrening vs fine-tuning.** Ciało było już wytrenowane (pretrening) na ogromnych zbiorach tekstu. My robimy tylko **fine-tuning** (douczanie) pod swoje zadanie, tak jak wykształconego człowieka wystarczy krótko przeszkolić do konkretnej pracy.

**Etykieta `-100`.** Umowna wartość znaczy: "tego miejsca nie licz do straty". Używamy jej np. dla paddingu.

## BERT a GPT-2 w jednym zdaniu

- **BERT** czyta cały tekst naraz, w obie strony (jak człowiek, który widzi całą kartkę). Świetny do *rozumienia* tekstu: klasyfikacji, rozpoznawania nazw.
- **GPT-2** pisze tekst słowo po słowie i widzi tylko to, co już napisał (jak ktoś, kto zasłonił prawą część kartki). Świetny do *generowania* tekstu.

In [61]:
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import peft
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BertForMaskedLM,
    BertForSequenceClassification,
    BertForTokenClassification,
    BertModel,
    DataCollatorWithPadding,
    GPT2ForSequenceClassification,
    GPT2ForTokenClassification,
    GPT2LMHeadModel,
    GPT2Model,
    Trainer,
    TrainingArguments,
)

transformers.logging.set_verbosity_error()  # ukrywa ostrzeżenia o "nowo zainicjowanych wagach" (wyjaśniamy je niżej)
torch.manual_seed(0)

DEVICE = (
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(
    f"transformers {transformers.__version__} | peft {peft.__version__} | torch {torch.__version__} | urządzenie do treningu: {DEVICE}"
)


def n_params(module, trainable_only=False):
    return sum(
        p.numel() for p in module.parameters() if p.requires_grad or not trainable_only
    )


def check(name, predicted, actual):
    """Porównuje Twoją prognozę z prawdziwą wartością; pusta prognoza (None) po prostu pokazuje odpowiedź."""
    if predicted is None:
        print(f"[{name}] brak prognozy. Prawdziwa wartość: {actual}")
    else:
        verdict = "dobrze" if tuple(predicted) == tuple(actual) else "ŹLE"
        print(
            f"[{name}] Twoja odpowiedź: {tuple(predicted)} -> {verdict}. Prawdziwa wartość: {tuple(actual)}"
        )


transformers 5.19.0 | peft 0.21.2 | torch 2.14.1 | urządzenie do treningu: mps


# Część A
## Klasy BERT w bibliotece `transformers`

**W prostych słowach.** BERT to model, który "przeczytał" ogromną ilość tekstu i dzięki temu zna język. W bibliotece `transformers` występuje w kilku wersjach. To nadal ten sam "mózg" (`BertModel`), tylko z różnymi **nakładkami** (nazywanymi *głowicami*, ang. *head*) do różnych zadań:

| Klasa | Co robi (po ludzku) | Przykład zadania |
|---|---|---|
| `BertModel` | sam "mózg": zamienia tekst na liczby opisujące jego znaczenie | (nic nie przewiduje, daje tylko wektory) |
| `BertForMaskedLM` | zgaduje **brakujące słowo** w zdaniu | "Stolicą Francji jest [MASK]." -> "Paryż" |
| `BertForSequenceClassification` | daje **jedną etykietę całemu tekstowi** | opis objawów -> diagnoza; recenzja -> pozytywna/negatywna |
| `BertForTokenClassification` | daje **etykietę każdemu słowu** (tokenowi) | "Jan Kowalski mieszka w Warszawie" -> osoba, osoba, -, -, miejsce |

```text
tekst -> tokenizer -> liczby (input_ids, attention_mask, token_type_ids)
                          |
                    BertModel ("mózg")
                          |
        +-----------------+------------------+-------------------+
        | ForMaskedLM     | ForSequenceClass.| ForTokenClass.    |
        | każdy token ->  | całe zdanie ->   | każdy token ->    |
        | słowo ze słown. | jedna etykieta   | własna etykieta   |
        | wynik (B,T,V)   | wynik (B,L)      | wynik (B,T,L)     |
        +-----------------+------------------+-------------------+
```

Kluczowa myśl całej części A: **mózg jest wspólny, różni się tylko mała głowica na końcu** oraz to, *z których wektorów* ona korzysta.

### A.1 Od tekstu do liczb

Model nie rozumie liter, tylko liczby. Tokenizer zamienia tekst na trzy "tabelki" o tym samym rozmiarze:

- `input_ids`: numery tokenów (tokeny to słowa lub kawałki słów). Na początku zawsze `[CLS]`, na końcu `[SEP]`.
- `attention_mask`: `1` = prawdziwy token, `0` = wypełniacz (padding). Dzięki temu model wie, które miejsca ignorować.
- `token_type_ids`: który to fragment tekstu (zdanie A = 0, zdanie B = 1). Przy jednym zdaniu same zera.

**Po co padding?** Żeby przetwarzać kilka zdań naraz, muszą mieć tę samą długość, więc krótsze "dopychamy" tokenami `[PAD]`.

In [41]:
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

sentences = ["The patient has a high fever.", "Hello world"]
batch = tok(sentences, padding=True, return_tensors="pt")

for key, value in batch.items():
    print(f"{key:15s} kształt={tuple(value.shape)}")
    print(value, "\n")
print(tok.convert_ids_to_tokens(batch["input_ids"][0]))
print(tok.convert_ids_to_tokens(batch["input_ids"][1]))


input_ids       kształt=(2, 9)
tensor([[ 101, 1996, 5776, 2038, 1037, 2152, 9016, 1012,  102],
        [ 101, 7592, 2088,  102,    0,    0,    0,    0,    0]]) 

token_type_ids  kształt=(2, 9)
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]) 

attention_mask  kształt=(2, 9)
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0, 0]]) 

['[CLS]', 'the', 'patient', 'has', 'a', 'high', 'fever', '.', '[SEP]']
['[CLS]', 'hello', 'world', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


### A.2 `BertModel`: "mózg" BERT-a

Wyobraź sobie linię produkcyjną z trzema stanowiskami. Każde jest osobnym elementem modelu, który możesz podejrzeć w kodzie:

1. **`embeddings` (osadzenia):** każdy token (numer) zamieniany jest na wektor 768 liczb. Do niego dodawana jest informacja o *pozycji* tokenu w zdaniu i o tym, z którego zdania pochodzi.
2. **`encoder.layer[i]` (12 identycznych warstw):** serce modelu. Każda warstwa robi dwie rzeczy:
   - **self-attention (uwaga):** każdy token "zerka" na pozostałe tokeny i uzupełnia swoje znaczenie o kontekst. Np. słowo "zamek" obok "drzwi" dostaje inny wektor niż obok "Malbork";
   - małą sieć neuronową, która przetwarza wynik (rozmiar 768 -> 3072 -> 768).
   
   W BERT-cie uwaga jest **dwukierunkowa**: token widzi słowa przed sobą *i* po sobie (ale nie widzi paddingu).
3. **`pooler` (opcjonalny):** bierze wektor tokenu `[CLS]` (stoi zawsze na początku i z założenia "zbiera sens całego zdania") i robi z niego jedno "streszczenie zdania". To jedyny powód, dla którego istnieje `pooler_output`.

**Wynik:** `last_hidden_state` ma po jednym wektorze na token, czyli kształt `(B, T, H)` (zdania x tokeny x 768 liczb). `pooler_output` ma po jednym wektorze na zdanie: `(B, H)`.

Zajrzyj w prawdziwe moduły i zobacz, gdzie są parametry (liczby, które model "wyuczył się" podczas treningu).

In [42]:
bert = BertModel.from_pretrained("bert-base-uncased").eval()

cfg = bert.config
print(
    {
        k: getattr(cfg, k)
        for k in [
            "hidden_size",
            "num_hidden_layers",
            "num_attention_heads",
            "intermediate_size",
            "vocab_size",
            "max_position_embeddings",
        ]
    }
)
print()
for name, child in bert.named_children():
    print(f"{name:12s} {n_params(child):>12,} parametrów")
print(f"{'RAZEM':12s} {n_params(bert):>12,} parametrów\n")

print(bert.embeddings, "\n")
print(bert.encoder.layer[0], "\n")
print(bert.pooler)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'hidden_size': 768, 'num_hidden_layers': 12, 'num_attention_heads': 12, 'intermediate_size': 3072, 'vocab_size': 30522, 'max_position_embeddings': 512}

embeddings     23,837,184 parametrów
encoder        85,054,464 parametrów
pooler            590,592 parametrów
RAZEM         109,482,240 parametrów

BertEmbeddings(
  (word_embeddings): Embedding(30522, 768, padding_idx=0)
  (position_embeddings): Embedding(512, 768)
  (token_type_embeddings): Embedding(2, 768)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
) 

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_featur

**Przewidź.** Dla `batch` powyżej (2 zdania, z paddingiem): jakie kształty będą miały `last_hidden_state` i `pooler_output`? Podpowiedź: `T` odczytasz z wypisanych tokenów w A.1 (liczba tokenów w dłuższym zdaniu).

In [43]:
pred_last_hidden = (2, 9, 768)  # np. (B, T, H) z wpisanymi liczbami
pred_pooled = (2, 768)  # np. (B, H) z wpisanymi liczbami


In [44]:
with torch.no_grad():
    out = bert(**batch)

check("last_hidden_state", pred_last_hidden, out.last_hidden_state.shape)
check("pooler_output", pred_pooled, out.pooler_output.shape)
print("klucze wyniku:", list(out.keys()))


[last_hidden_state] Twoja odpowiedź: (2, 9, 768) -> dobrze. Prawdziwa wartość: (2, 9, 768)
[pooler_output] Twoja odpowiedź: (2, 768) -> dobrze. Prawdziwa wartość: (2, 768)
klucze wyniku: ['last_hidden_state', 'pooler_output']


### A.3 Dwie rzeczy, które warto sprawdzić samemu

1. `pooler_output` to **nie** magiczny "wektor zdania". To po prostu `tanh(Linear(h_[CLS]))` i nic więcej. Sprawdzimy to ręcznie.
2. Padding nie może wpływać na prawdziwe tokeny. `attention_mask` sprawia, że wypełniacze dostają po przejściu przez softmax praktycznie zerową uwagę. Zdanie wrzucone samo i zdanie wrzucone w paczce z paddingiem powinno więc dać (prawie) te same wektory.

In [45]:
# 1) pooler_output == tanh(dense(h[CLS]))
manual_pooled = torch.tanh(bert.pooler.dense(out.last_hidden_state[:, 0]))
print(
    "pooler = tanh(dense(CLS)):",
    torch.allclose(manual_pooled, out.pooler_output, atol=1e-6),
)

# 2) zdanie samo vs. to samo zdanie w paczce z paddingiem
alone = tok(sentences[1], return_tensors="pt")
n_real = alone["input_ids"].shape[1]
with torch.no_grad():
    out_alone = bert(**alone)
print(
    "te same wektory z paddingiem i bez:",
    torch.allclose(
        out_alone.last_hidden_state[0], out.last_hidden_state[1, :n_real], atol=1e-4
    ),
)

# żeby zobaczyć wagi uwagi, potrzebna jest implementacja "eager" (domyślna szybka ich nie zwraca)
bert_eager = BertModel.from_pretrained(
    "bert-base-uncased", attn_implementation="eager"
).eval()
with torch.no_grad():
    attentions = bert_eager(**batch, output_attentions=True).attentions
print(
    f"\nwagi uwagi: {len(attentions)} warstw, każda o kształcie {tuple(attentions[0].shape)} = (B, głowice, T, T)"
)
print(
    "największa uwaga, jaką jakikolwiek token daje paddingowi (zdanie 2):",
    attentions[0][1, :, :, n_real:].max().item(),
)


pooler = tanh(dense(CLS)): True
te same wektory z paddingiem i bez: True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


wagi uwagi: 12 warstw, każda o kształcie (2, 12, 9, 9) = (B, głowice, T, T)
największa uwaga, jaką jakikolwiek token daje paddingowi (zdanie 2): 0.0


### A.4 `BertForMaskedLM`: zgadywanie brakującego słowa

**W prostych słowach.** To "uzupełnianka": zasłaniasz słowo tokenem `[MASK]`, a model zgaduje, co tam pasuje. Tak BERT był uczony na ogromnych tekstach (stąd w ogóle zna język).

Budowa: "mózg" `BertModel` (bez poolera) + głowica `cls`, która dla *każdego* tokenu daje wynik dla każdego słowa ze słownika:

`wektor tokenu (H) -> mała warstwa przetwarzająca -> warstwa wyjściowa (H -> V) -> logity (B, T, V)`

- Czyta wektory **wszystkich** tokenów (nie poolera), bo zasłonić można dowolne słowo.
- Wagi warstwy wyjściowej są **te same** co wagi osadzeń wejściowych (*weight tying*). Model oszczędza pamięć, a słowo "na wejściu" i "na wyjściu" ma spójną reprezentację.
- **Strata** liczona jest tylko tam, gdzie etykieta jest różna od `-100`. Tak uczy się "zgadywać tylko zasłonięte 15% słów": wszędzie indziej dajemy `-100`.
- Przyda się w zadaniu 4 (augmentacja danych): zasłaniasz słowa w zdaniu treningowym i prosisz model o zamienniki.

In [46]:
mlm = BertForMaskedLM.from_pretrained("bert-base-uncased").eval()
print(mlm.cls, "\n")

word_emb = mlm.bert.embeddings.word_embeddings.weight
decoder_w = mlm.cls.predictions.decoder.weight
print(
    "wagi wyjściowe są tym samym tensorem co osadzenia wejściowe:",
    decoder_w.data_ptr() == word_emb.data_ptr(),
)
print("czy ma pooler:", mlm.bert.pooler is not None)

text = f"The capital of France is {tok.mask_token}."
enc = tok(text, return_tensors="pt")
mask_pos = (enc["input_ids"][0] == tok.mask_token_id).nonzero().item()
with torch.no_grad():
    logits = mlm(**enc).logits
print("\nkształt logitów:", tuple(logits.shape), "= (B, T, V)")
top = logits[0, mask_pos].softmax(-1).topk(5)
print("5 najlepszych propozycji na miejsce [MASK]:")
for p, i in zip(top.values, top.indices):
    print(f"  {tok.convert_ids_to_tokens(i.item()):12s} {p.item():.3f}")

# strata tylko na zasłoniętym miejscu: wszędzie indziej -100
labels = torch.full_like(enc["input_ids"], -100)
labels[0, mask_pos] = tok.convert_tokens_to_ids("paris")
with torch.no_grad():
    loss = mlm(**enc, labels=labels).loss
print("\nstrata dla prawidłowej odpowiedzi 'paris':", round(loss.item(), 3))


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertOnlyMLMHead(
  (predictions): BertLMPredictionHead(
    (transform): BertPredictionHeadTransform(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (transform_act_fn): GELUActivation()
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    )
    (decoder): Linear(in_features=768, out_features=30522, bias=True)
  )
) 

wagi wyjściowe są tym samym tensorem co osadzenia wejściowe: True
czy ma pooler: False

kształt logitów: (1, 9, 30522) = (B, T, V)
5 najlepszych propozycji na miejsce [MASK]:
  paris        0.417
  lille        0.071
  lyon         0.063
  marseille    0.044
  tours        0.030

strata dla prawidłowej odpowiedzi 'paris': 0.875


### A.5 `BertForSequenceClassification`: jedna etykieta na cały tekst

**W prostych słowach.** Dajesz całe zdanie, dostajesz jedną odpowiedź (np. diagnozę). Model bierze "streszczenie zdania" (`pooler_output`) i podaje je do jednej małej warstwy, która zamienia 768 liczb na `L` wyników, po jednym na każdą klasę.

`BertModel` (z poolerem) -> `dropout` -> `classifier = Linear(H, L)` na `pooler_output`.

- *Dropout* to losowe "wyłączanie" części liczb podczas treningu, żeby model się nie przeuczył. W trybie `.eval()` nie robi nic.
- Warstwa `classifier` jest **losowa na starcie**. To właśnie ostrzeżenie "niektóre wagi zostały nowo zainicjowane", które mogłeś widzieć. Dopóki jej nie douczymy, odpowiedzi są losowe. "Mózg" jest dobry, ale głowica jeszcze nie wie, czego od niej chcemy.
- Strata zależy od typu etykiet: liczby całkowite i `L > 1` -> entropia krzyżowa (zwykła klasyfikacja); `L == 1` -> błąd średniokwadratowy (regresja); etykiety wielokrotne (0/1 dla każdej klasy) -> BCE. Ustawiane przez `config.problem_type`.
- W Części C i zadaniu 1 to ta klasa dostanie adapter PEFT.

In [ ]:
seq = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=3
).eval()
print(seq.dropout, seq.classifier, "\n")

with torch.no_grad():
    out_seq = seq(**batch, labels=torch.tensor([0, 2]))
    pooled = seq.bert(**batch).pooler_output

print("logity:", tuple(out_seq.logits.shape), "= (B, L)")
manual_logits = seq.classifier(
    seq.dropout(pooled)
)  # w trybie eval dropout nic nie zmienia
print(
    "logity == classifier(pooler_output):",
    torch.allclose(manual_logits, out_seq.logits, atol=1e-6),
)
print(
    "strata == cross_entropy:",
    torch.allclose(F.cross_entropy(out_seq.logits, torch.tensor([0, 2])), out_seq.loss),
)
print(
    "parametry głowicy:",
    n_params(seq) - n_params(seq.bert),
    "(= H*L + L =",
    768 * 3 + 3,
    ")",
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dropout(p=0.1, inplace=False) Linear(in_features=768, out_features=3, bias=True) 

logity: (2, 3) = (B, L)
logity == classifier(pooler_output): True
strata == cross_entropy: True
parametry głowicy: 2307 (= H*L + L = 2307 )


### A.6 `BertForTokenClassification`: etykieta dla każdego tokenu

**W prostych słowach.** Zamiast jednej odpowiedzi na całe zdanie, model daje odpowiedź **dla każdego słowa**. Np. rozpoznawanie nazw: które słowa to osoba, które miejsce.

`BertModel` (bez poolera) -> `dropout` -> `classifier = Linear(H, L)` stosowany do **każdego** tokenu z `last_hidden_state`. Wynik: `(B, T, L)`.

Trudność praktyczna to **dopasowanie etykiet**. Etykiety mamy dla *słów*, ale model pracuje na *tokenach* (jedno słowo może się rozpaść na kilka), do tego dochodzą `[CLS]`, `[SEP]` i padding. Standardowa umowa:

- etykietę dajemy tylko **pierwszemu** tokenowi każdego słowa, pozostałe kawałki dostają `-100`;
- `[CLS]`, `[SEP]` i padding też dostają `-100`.

Funkcja straty `CrossEntropyLoss(ignore_index=-100)` pomija wtedy te miejsca. (To samo "maskowanie" wraca w zadaniu 3 dla GPT-2.)

In [ ]:
tokcls = BertForTokenClassification.from_pretrained(
    "bert-base-uncased", num_labels=5
).eval()
print(tokcls.dropout, tokcls.classifier)
print("czy ma pooler:", tokcls.bert.pooler is not None, "\n")

words = ["Jan", "Kowalski", "lives", "in", "Warsaw"]
word_labels = [1, 2, 0, 0, 3]  # np. B-OSOBA, I-OSOBA, O, O, B-MIEJSCE
enc = tok(words, is_split_into_words=True, return_tensors="pt")

aligned, prev = [], None
for wid in enc.word_ids():
    if (
        wid is None or wid == prev
    ):  # token specjalny albo kolejny kawałek tego samego słowa
        aligned.append(-100)
    else:
        aligned.append(word_labels[wid])
    prev = wid
labels = torch.tensor([aligned])

print(
    pd.DataFrame(
        {
            "token": tok.convert_ids_to_tokens(enc["input_ids"][0]),
            "id_słowa": enc.word_ids(),
            "etykieta": aligned,
        }
    ).T.to_string(header=False),
    "\n",
)

with torch.no_grad():
    out_tok = tokcls(**enc, labels=labels)
print("logity:", tuple(out_tok.logits.shape), "= (B, T, L)")
manual_loss = nn.CrossEntropyLoss(ignore_index=-100)(
    out_tok.logits.view(-1, 5), labels.view(-1)
)
print(
    "strata == CrossEntropyLoss(ignore_index=-100):",
    torch.allclose(manual_loss, out_tok.loss),
)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Dropout(p=0.1, inplace=False) Linear(in_features=768, out_features=5, bias=True)
czy ma pooler: False 

token     [CLS]  jan  kowalski  lives   in  warsaw  [SEP]
id_słowa    NaN  0.0       1.0    2.0  3.0     4.0    NaN
etykieta   -100    1         2      0    0       3   -100 

logity: (1, 7, 5) = (B, T, L)
strata == CrossEntropyLoss(ignore_index=-100): True


### A.7 To samo ciało, różne głowice

Sprawdźmy główną tezę części A: wszystkie cztery klasy ładują **te same wytrenowane wagi "mózgu"**, a różni je tylko głowica.

Zwróć uwagę na tabelę: głowica klasyfikacji ma kilka tysięcy parametrów, a "mózg" ponad 100 milionów. Głowica jest maleńka w porównaniu z resztą.

In [ ]:
def q_weight(m):
    return m.encoder.layer[0].attention.self.query.weight


bodies = {
    "BertModel": bert,
    "ForMaskedLM": mlm.bert,
    "ForSequenceClassification": seq.bert,
    "ForTokenClassification": tokcls.bert,
}
print(
    "identyczne wagi query w warstwie 0:",
    all(torch.equal(q_weight(bert), q_weight(b)) for b in bodies.values()),
)

# wagi wyjściowe MLM są wspólne z osadzeniami, więc parameters() liczy je raz (w "ciele")
rows = [
    {"klasa": "BertModel", "parametry ciała": n_params(bert), "parametry głowicy": 0}
]
for name, model in [
    ("BertForMaskedLM", mlm),
    ("BertForSequenceClassification", seq),
    ("BertForTokenClassification", tokcls),
]:
    rows.append(
        {
            "klasa": name,
            "parametry ciała": n_params(model.bert),
            "parametry głowicy": n_params(model) - n_params(model.bert),
        }
    )
pd.DataFrame(rows).style.format(
    {"parametry ciała": "{:,}", "parametry głowicy": "{:,}"}
)


identyczne wagi query w warstwie 0: True


,klasa,parametry ciała,parametry głowicy
0,BertModel,"109,482,240",0
1,BertForMaskedLM,"108,891,648","622,650"
2,BertForSequenceClassification,"109,482,240","2,307"
3,BertForTokenClassification,"108,891,648","3,845"


**Twoja kolej (A).** Zadanie 2 każe użyć wektora `[CLS]` jako reprezentacji tekstu. Popularną alternatywą jest **średnia z wektorów prawdziwych tokenów** (*mean pooling*). Zaimplementuj `mean_pool(last_hidden_state, attention_mask)` tak, żeby padding **nie** wchodził do średniej. Komórka niżej porównuje wynik z wersją referencyjną w pętli. Zastanów się, dlaczego zwykłe `.mean(1)` byłoby błędne dla naszej paczki z paddingiem.

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)  # (B, T, 1)
    summed = (last_hidden_state * mask).sum(dim=1)  # padding zerowany przez maskę
    return summed / mask.sum(
        dim=1
    )  # dzielimy przez liczbę prawdziwych tokenów, nie przez T


In [77]:
reference = torch.stack(
    [
        out.last_hidden_state[i, : int(batch["attention_mask"][i].sum())].mean(0)
        for i in range(2)
    ]
)
mine = mean_pool(out.last_hidden_state, batch["attention_mask"])
if mine is None:
    print("jeszcze niezaimplementowane")
else:
    print("zgodne z wersją referencyjną:", torch.allclose(mine, reference, atol=1e-5))
    print(
        "naiwne .mean(1) zgodne z referencją:",
        torch.allclose(out.last_hidden_state.mean(1), reference, atol=1e-5),
    )


zgodne z wersją referencyjną: True
naiwne .mean(1) zgodne z referencją: False


# Część B
## PEFT: douczanie małym kosztem

**Problem.** Pełne douczanie (fine-tuning) zmienia *wszystkie* parametry. BERT-base ma ich ok. 110 mln, więc do każdego zadania trzeba trzymać własną pełną kopię modelu (~440 MB), a do tego pamięć na dane optymalizatora (Adam trzyma dodatkowo ok. 2 liczby na każdy parametr). Dla dużych modeli jest to bardzo drogie.

**Pomysł PEFT** (*Parameter-Efficient Fine-Tuning*). Zamroź wytrenowany model i naucz tylko **bardzo mały** dodatek.

**Analogia.** Masz grubą, drukowaną książkę (model). Pełny fine-tuning to przepisywanie jej całej. PEFT to zostawienie książki w spokoju i dopisanie **małych karteczek na marginesach**. Karteczki są tanie, można je trzymać osobno dla każdego zadania i dołączać do tej samej książki.

### LoRA: jak to działa

**LoRA** (*Low-Rank Adaptation*) to metoda PEFT używana w całym labie. W modelu są duże macierze wag $W$ (np. $768 \times 768$ = 589 824 liczby). LoRA zostawia $W$ zamrożone i dodaje do jej wyniku małą poprawkę, która jest iloczynem dwóch "chudych" macierzy $B$ i $A$:

$$h = W x + \frac{\alpha}{r} B A x, \qquad A \in \mathbb{R}^{r \times d_{in}},\; B \in \mathbb{R}^{d_{out} \times r}$$

Intuicja: zamiast uczyć jedną ogromną tabelę $768 \times 768$, uczymy dwie wąskie: $768 \times r$ i $r \times 768$. Liczba $r$ (**ranga**) to "szerokość przejścia" między nimi, np. $r = 8$.

- **Oszczędność:** $768 \cdot 768 = 589\,824$ parametrów kontra $2 \cdot 768 \cdot 8 = 12\,288$ (ok. 2%).
- `B` startuje od **zera**, więc na początku poprawka jest zerowa i model jest *dokładnie* taki jak oryginał. Karteczki są puste, trening je stopniowo zapisuje.
- Po treningu $W + \frac{\alpha}{r} BA$ można **scalić** w jedną macierz. Po scaleniu model nie jest wolniejszy.
- Najważniejsze ustawienia: `r` (ranga), `lora_alpha` (siła poprawki, mnożnik to `alpha / r`), `lora_dropout`, `target_modules` (w których warstwach dodajemy poprawki).

### B.1 LoRA od zera (kilkanaście linijek)

Zanim użyjemy biblioteki, zbudujmy mechanizm sami na jednej warstwie `Linear`. Wtedy "magia" PEFT przestaje być magią.

In [52]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int = 8, alpha: int = 16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False  # oryginalna warstwa zamrożona
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(
            torch.zeros(base.out_features, r)
        )  # zerowa inicjalizacja => na starcie wynik identyczny z oryginałem
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale


base_linear = nn.Linear(768, 768)
lora = LoRALinear(base_linear, r=8, alpha=16)
x = torch.randn(4, 768)

print("na starcie identyczne z oryginałem:", torch.allclose(lora(x), base_linear(x)))
print(
    f"uczone: {n_params(lora, trainable_only=True):,} z {n_params(lora):,} ({100 * n_params(lora, True) / n_params(lora):.2f}%)"
)

with torch.no_grad():
    lora.B.normal_(std=0.01)  # udajemy, że trening coś zmienił
merged_weight = base_linear.weight + lora.scale * lora.B @ lora.A
print(
    "po 'treningu' różni się od oryginału:",
    not torch.allclose(lora(x), base_linear(x)),
)
print(
    "scalone W + (alpha/r)BA daje ten sam wynik:",
    torch.allclose(lora(x), F.linear(x, merged_weight, base_linear.bias), atol=1e-5),
)


na starcie identyczne z oryginałem: True
uczone: 12,288 z 602,880 (2.04%)
po 'treningu' różni się od oryginału: True
scalone W + (alpha/r)BA daje ten sam wynik: True


### B.2 Biblioteka PEFT na BERT-cie

Schemat z quicktour to cztery kroki:

1. Wczytaj model bazowy (`BertForSequenceClassification`).
2. Opisz adapter w konfiguracji (`LoraConfig`): `task_type` mówi PEFT-owi, jakiej głowicy używa zadanie, a `target_modules`, w których warstwach `Linear` dodać poprawki LoRA.
3. `get_peft_model(model, config)` zamraża model bazowy i wstrzykuje uczone macierze LoRA.
4. Trenuj jak zwykle; `save_pretrained` zapisuje **tylko adapter** (kilka MB zamiast setek).

Dwa szczegóły, które mają znaczenie w praktyce:

- Przy `task_type=SEQ_CLS` (albo `TOKEN_CLS`) PEFT sam dodaje głowicę (`classifier`) do `modules_to_save`, czyli **uczy ją w całości**, a resztę modelu zamraża. Ma to sens: głowica jest losowa na starcie, więc zamrożenie jej byłoby bez sensu.
- Jeśli pominiesz `target_modules`, PEFT użyje domyślnych dla danej architektury (dla BERT-a: `query` i `value`, czyli dwie z trzech warstw uwagi).

In [80]:
base_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=3
)
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
)
peft_model = get_peft_model(base_model, lora_cfg)
peft_model.print_trainable_parameters()

trainable_names = [n for n, p in peft_model.named_parameters() if p.requires_grad]
print("\nliczba uczonych tensorów:", len(trainable_names))
print("pierwsze 4:", *trainable_names[:4], sep="\n  ")
print(
    "uczone, ale to nie LoRA:",
    *[n for n in trainable_names if "lora_" not in n],
    sep="\n  ",
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707

liczba uczonych tensorów: 50
pierwsze 4:
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_A.default.weight
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_B.default.weight
  base_model.model.bert.encoder.layer.0.attention.self.value.lora_A.default.weight
  base_model.model.bert.encoder.layer.0.attention.self.value.lora_B.default.weight
uczone, ale to nie LoRA:
  base_model.model.classifier.modules_to_save.default.weight
  base_model.model.classifier.modules_to_save.default.bias


**Przewidź.** Konfiguracja powyżej dodaje LoRA do `query` i `value` w każdej z 12 warstw BERT-base, z `r=8`.

1. Ile parametrów uczonych pochodzi tylko z macierzy LoRA? (Na każdą adaptowaną warstwę `Linear` przypadają dwie macierze: `A` i `B`; każda warstwa ma rozmiar 768 -> 768.)
2. Ile parametrów uczonych pochodzi z głowicy klasyfikacji (`H -> L=3`, z biasem)?

Podpowiedź: policz najpierw, ile parametrów ma jedna para `A`+`B`, potem ile jest takich par.

In [ ]:
pred_lora_params = 2 * 12 * 2 * 8 * 768  # jedna liczba całkowita
pred_head_params = 768 * 3 + 3  # jedna liczba całkowita


In [94]:
actual_lora = sum(
    p.numel()
    for n, p in peft_model.named_parameters()
    if p.requires_grad and "lora_" in n
)
actual_head = sum(
    p.numel()
    for n, p in peft_model.named_parameters()
    if p.requires_grad and "lora_" not in n
)
check(
    "parametry LoRA",
    None if pred_lora_params is None else (pred_lora_params,),
    (actual_lora,),
)
check(
    "parametry głowicy",
    None if pred_head_params is None else (pred_head_params,),
    (actual_head,),
)


[parametry LoRA] Twoja odpowiedź: (294912,) -> dobrze. Prawdziwa wartość: (294912,)
[parametry głowicy] Twoja odpowiedź: (2307,) -> dobrze. Prawdziwa wartość: (2307,)


### B.3 Jak zachowuje się adapter: start, wyłączanie, zapis, wczytanie, scalenie

Sprawdźmy na prawdziwym modelu to, o czym była mowa w teorii:

1. Na starcie `B = 0`, więc model z adapterem = model bazowy (adapter włączony == wyłączony).
2. Gdy zmienimy `B`, wyniki się różnią; `disable_adapter()` przywraca zachowanie modelu bazowego.
3. `save_pretrained` zapisuje tylko adapter (plus wytrenowaną głowicę). Porównaj rozmiar z całym modelem.
4. `PeftModel.from_pretrained` podłącza adapter z powrotem do świeżego modelu bazowego.
5. `merge_and_unload()` wtapia poprawkę w wagi i zwraca zwykły model z `transformers`.

In [95]:
peft_model.eval()


def logits_of(model):
    with torch.no_grad():
        return model(**batch).logits


# 1) adapter włączony vs wyłączony na starcie
on = logits_of(peft_model)
with peft_model.disable_adapter():
    off = logits_of(peft_model)
print(
    "1) na starcie adapter włączony == wyłączony:", torch.allclose(on, off, atol=1e-5)
)

# 2) zmieniamy B w każdej warstwie LoRA
with torch.no_grad():
    for n, p in peft_model.named_parameters():
        if "lora_B" in n:
            p.normal_(std=0.05)
changed = logits_of(peft_model)
with peft_model.disable_adapter():
    off_after = logits_of(peft_model)
print(
    "2) po zmianie B wynik się różni:",
    not torch.allclose(changed, on, atol=1e-3),
    "| adapter wyłączony nadal daje wynik bazowy:",
    torch.allclose(off_after, off, atol=1e-5),
)

# 3) + 4) zapis samego adaptera i wczytanie na świeży model bazowy
with tempfile.TemporaryDirectory() as d:
    peft_model.save_pretrained(d)
    sizes = {f: os.path.getsize(os.path.join(d, f)) / 1e6 for f in os.listdir(d)}
    print("3) zapisane pliki (MB):", {k: round(v, 2) for k, v in sizes.items()})
    fresh_base = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=3
    )
    reloaded = PeftModel.from_pretrained(fresh_base, d).eval()
print("   cały model miałby ok.", round(n_params(base_model) * 4 / 1e6), "MB (fp32)")
print(
    "4) wczytany adapter daje te same logity:",
    torch.allclose(logits_of(reloaded), changed, atol=1e-4),
)

# 5) scalenie z wagami
merged = peft_model.merge_and_unload().eval()
print(
    "5) typ scalonego modelu:",
    type(merged).__name__,
    "| te same logity:",
    torch.allclose(logits_of(merged), changed, atol=1e-4),
)


1) na starcie adapter włączony == wyłączony: True
2) po zmianie B wynik się różni: True | adapter wyłączony nadal daje wynik bazowy: True
3) zapisane pliki (MB): {'adapter_model.safetensors': 1.2, 'README.md': 0.01, 'adapter_config.json': 0.0}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   cały model miałby ok. 439 MB (fp32)
4) wczytany adapter daje te same logity: True
5) typ scalonego modelu: BertForSequenceClassification | te same logity: True


### B.4 Dokumentacja i przykłady: co przeczytać

**Quicktour** (<https://huggingface.co/docs/peft/quicktour>): przeczytaj pod kątem obiektów, których użyliśmy wyżej: `LoraConfig`, `get_peft_model`, `print_trainable_parameters`, `save_pretrained`, `PeftModel.from_pretrained` oraz `AutoPeftModelForCausalLM`.

**Repozytorium z przykładami** (<https://github.com/huggingface/peft/tree/main/examples>): w tym laboratorium przydadzą się foldery
- `sequence_classification`: wzorzec do Części C i zadania 1a (modele typu BERT, LoRA, `Trainer` lub własna pętla),
- `token_classification`: wzorzec do zadania 1b, w tym dopasowanie etykiet `-100` z A.6,
- `causal_language_modeling`: modele typu GPT-2, przydatne do zadania 3 i generowania tekstu.

Pytania, na które warto odpowiedzieć podczas czytania (przydadzą się w zadaniu 1):
1. Jakie `target_modules` używa każdy przykład i dlaczego?
2. Gdzie ustawiają `task_type` i co by się zepsuło, gdyby go nie podali?
3. Jak radzą sobie z `pad_token` w modelach typu GPT?
4. Gdzie zapisywany jest adapter i co jest w tym katalogu?

W repozytorium są też inne rodziny adapterów (`dora_finetuning`, `olora_finetuning` itd.). To warianty tego samego interfejsu `get_peft_model`; na potrzeby labu wystarczy LoRA.

# Część C
## Powtórzenie tutoriala z Medium: PEFT-BERT, objawy -> diagnoza

Źródło: [Parameter-Efficient Fine-Tuning (PEFT) of BERT base model to predict medical diagnosis](https://medium.com/@nubyra/parameter-efficient-fine-tuning-peft-of-bertbase-model-to-predict-medical-diagnosis-5086a1828f4b).

**Cel w jednym zdaniu:** dajemy modelowi opis objawów (po angielsku), a on ma zgadnąć jedną z 22 chorób. Wykorzystujemy do tego wszystko, co poznałeś w częściach A i B: `BertForSequenceClassification` + adapter LoRA.

Kolejne kroki tutoriala (komórki poniżej idą dokładnie tym tokiem):

1. Wczytaj zbiór `gretelai/symptom_to_diagnosis` (853 teksty treningowe, 212 testowych, 22 diagnozy).
2. **Punkt odniesienia:** `bert-base-uncased` z nieuczoną głowicą klasyfikacji. Tutorial podaje **4,25%** trafności, czyli poziom zgadywania (przy 22 klasach losowo trafiłbyś ok. 4,5%).
3. Dodaj LoRA (`r=64, lora_alpha=1, lora_dropout=0.1`). Tutorial podaje **2 376 214** uczonych parametrów (2,12%).
4. Trenuj 15 epok przez `Trainer` (*epoka* = jedno przejście przez cały zbiór treningowy; tempo uczenia `2e-3`, paczka 4 zdania, `weight_decay` 0,01). Tutorial podaje **94,34%** trafności.
5. Zapisz adapter, wczytaj go na świeży model bazowy i zrób predykcje.

**Świadome różnice względem artykułu z 2024 r.** (zmieniły się API bibliotek, metoda jest ta sama):

| Artykuł | Tutaj | Powód |
|---|---|---|
| `evaluation_strategy=` | `eval_strategy=` | zmiana nazwy w obecnym `transformers` |
| `Trainer(tokenizer=...)` | `Trainer(processing_class=...)` | zmiana nazwy w obecnym `transformers` |
| `padding="max_length"` (512 tokenów) | przycinanie + padding dynamiczny przez `DataCollatorWithPadding` | ten sam wynik, dużo mniej obliczeń (teksty są krótkie) |
| wyniki w katalogu roboczym | wszystko w `02/artifacts/` | porządek w repozytorium |

Artykuł twierdzi, że LoRA jest dodana do *Query i Key*. Domyślnie PEFT dla BERT-a wybiera `query` i `value`. Po liczbie parametrów zobaczysz, co naprawdę użyto (sprawdzimy niżej).

### C.1 Dane

Każdy przykład to opis objawów (`input_text`) i diagnoza (`label`). Zbiór jest zbalansowany: mniej więcej tyle samo przykładów na każdą chorobę.

In [57]:
from datasets import load_dataset

ARTIFACTS = Path("..") / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

raw = load_dataset(
    "gretelai/symptom_to_diagnosis",
    data_files={"train": "train.jsonl", "test": "test.jsonl"},
)
raw = raw.rename_column("output_text", "label")
print(raw)
for ex in raw["train"].select(range(3)):
    print(f"\nWEJŚCIE: {ex['input_text']}\nETYKIETA: {ex['label']}")

labels_sorted = sorted(set(raw["train"]["label"]))
label2id = {l: i for i, l in enumerate(labels_sorted)}
id2label = {i: l for l, i in label2id.items()}
counts = pd.DataFrame(
    {
        "train": pd.Series(raw["train"]["label"]).value_counts(),
        "test": pd.Series(raw["test"]["label"]).value_counts(),
    }
).sort_index()
print(f"\n{len(label2id)} etykiet")
counts.T


DatasetDict({
    train: Dataset({
        features: ['label', 'input_text'],
        num_rows: 853
    })
    test: Dataset({
        features: ['label', 'input_text'],
        num_rows: 212
    })
})

WEJŚCIE: I've been having a lot of pain in my neck and back. I've also been having trouble with my balance and coordination. I've been coughing a lot and my limbs feel weak.
ETYKIETA: cervical spondylosis

WEJŚCIE: I have a rash on my face that is getting worse. It is red, inflamed, and has blisters that are bleeding clear pus. It is really painful.
ETYKIETA: impetigo

WEJŚCIE: I have been urinating blood. I sometimes feel sick to my stomach when I urinate. I often feel like I have a fever.
ETYKIETA: urinary tract infection

22 etykiet


,allergy,arthritis,bronchial asthma,cervical spondylosis,chicken pox,common cold,dengue,diabetes,drug reaction,fungal infection,...,impetigo,jaundice,malaria,migraine,peptic ulcer disease,pneumonia,psoriasis,typhoid,urinary tract infection,varicose veins
train,40,40,40,40,40,39,40,40,40,39,...,40,33,40,32,37,37,40,38,39,40
test,10,10,10,10,10,10,10,10,8,9,...,10,7,10,10,10,10,10,9,9,10


### C.2 Punkt odniesienia: nieuczona głowica

"Mózg" BERT-a już rozumie angielski, ale głowica klasyfikacji (ta mała warstwa z A.5) jest **losowa**. Model nie wie jeszcze, że ma wybierać choroby, więc przy 22 zbalansowanych klasach spodziewamy się trafności bliskiej $1/22 \approx 4{,}5\%$. To nasz punkt odniesienia, do którego porównamy wynik po douczeniu.

*Trafność (accuracy)* = jaki procent przykładów testowych model sklasyfikował poprawnie.

In [58]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
foundation = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=len(label2id), label2id=label2id, id2label=id2label
)


@torch.no_grad()
def predict_ids(model, texts, batch_size=32):
    """Zwraca numer klasy z najwyższym wynikiem dla każdego tekstu (działa dla zwykłych modeli i PEFT)."""
    model = model.to(DEVICE).eval()
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i : i + batch_size],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(DEVICE)
        preds.append(model(**enc).logits.argmax(-1).cpu())
    return torch.cat(preds).numpy()


test_labels = np.array([label2id[l] for l in raw["test"]["label"]])
baseline_acc = (
    predict_ids(foundation, raw["test"]["input_text"]) == test_labels
).mean() * 100
print(f"Trafność modelu bazowego: {baseline_acc:.2f}%  (artykuł: 4,25%)")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Trafność modelu bazowego: 6.60%  (artykuł: 4,25%)


### C.3 Konfiguracja LoRA (parametry z artykułu)

`r=64, lora_alpha=1, lora_dropout=0.1`. Skoro `lora_alpha / r = 1/64`, poprawka LoRA jest mocno "przyciszona". To ciekawy wybór: porównaj z `alpha/r = 2` z Części B.

**Przewidź.** Artykuł podaje 2 376 214 uczonych parametrów. Przy `r=64` i rozmiarze 768 jedna adaptowana warstwa `Linear` ma $2 \cdot 64 \cdot 768$ parametrów LoRA, a głowica klasyfikacji dla 22 etykiet ma $768 \cdot 22 + 22$. Ile adaptowanych warstw `Linear` to daje i które to są (`query`+`key`, `query`+`value`...)? Policz najpierw na kartce, a potem porównaj z tym, co PEFT wybiera domyślnie (poniżej).

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=64, lora_alpha=1, lora_dropout=0.1
)
med_model = get_peft_model(foundation, lora_config)

print(
    "target_modules wybrane przez PEFT:",
    med_model.peft_config["default"].target_modules,
)
med_model.print_trainable_parameters()
print(
    "artykuł:  trainable params: 2,376,214 || all params: 111,875,372 || trainable%: 2.1239831050572953"
)


target_modules wybrane przez PEFT: {'value', 'query'}
trainable params: 2,376,214 || all params: 111,875,372 || trainable%: 2.1240
artykuł:  trainable params: 2,376,214 || all params: 111,875,372 || trainable%: 2.1239831050572953


### C.4 Tokenizacja i trening

15 epok po 853 przykłady przy paczce 4 zdania to ok. 3200 kroków optymalizatora. Na GPU w Macu (`mps`) trwa to kilka minut. Podczas eksperymentów możesz zmniejszyć `EPOCHS`; w artykule jest 15.

Co dzieje się w jednym kroku treningu (warto to rozumieć):
1. Model dostaje paczkę 4 zdań i robi predykcję.
2. Liczymy **stratę**: jak bardzo się pomylił.
3. Optymalizator lekko zmienia *tylko uczone parametry* (macierze LoRA i głowicę), żeby strata była odrobinę mniejsza.

Po każdej epoce `Trainer` sprawdza trafność na zbiorze testowym (`eval_accuracy` w logu) i na końcu wczytuje najlepszą epokę.

In [60]:
EPOCHS = 15


def preprocess(batch):
    enc = tokenizer(
        batch["input_text"], truncation=True, max_length=128
    )  # padding dodaje później collator, osobno dla każdej paczki
    enc["label"] = [label2id[l] for l in batch["label"]]
    return enc


tokenized = raw.map(preprocess, batched=True, remove_columns=["input_text"])
lengths = [len(x) for x in tokenized["train"]["input_ids"]]
print(tokenized["train"])
print(
    f"długość w tokenach: max {max(lengths)}, średnia {np.mean(lengths):.1f}  (przycięcie do 128 dotknęło {sum(l == 128 for l in lengths)} przykładów)"
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (np.argmax(logits, axis=1) == labels).mean() * 100}


trainer = Trainer(
    model=med_model,
    args=TrainingArguments(
        output_dir=str(ARTIFACTS / "bert-lora-checkpoints"),
        learning_rate=2e-3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=32,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        report_to="none",
        seed=0,
    ),
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()


Map:   0%|          | 0/853 [00:00<?, ? examples/s]

Map:   0%|          | 0/212 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 853
})
długość w tokenach: max 76, średnia 39.5  (przycięcie do 128 dotknęło 0 przykładów)


/Users/mikolajmachalski/Studia/2sem/NLP/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '2.16', 'eval_accuracy': '37.26', 'eval_runtime': '1.241', 'eval_samples_per_second': '170.8', 'eval_steps_per_second': '5.641', 'epoch': '1'}


...

/Users/mikolajmachalski/Studia/2sem/NLP/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '0.1389', 'eval_accuracy': '95.75', 'eval_runtime': '1.205', 'eval_samples_per_second': '175.9', 'eval_steps_per_second': '5.808', 'epoch': '14'}


/Users/mikolajmachalski/Studia/2sem/NLP/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'loss': '0.008498', 'grad_norm': '0.004616', 'learning_rate': '0.0001315', 'epoch': '14.02'}
{'eval_loss': '0.1323', 'eval_accuracy': '95.28', 'eval_runtime': '1.191', 'eval_samples_per_second': '178', 'eval_steps_per_second': '5.876', 'epoch': '15'}
{'train_runtime': '253.9', 'train_samples_per_second': '50.39', 'train_steps_per_second': '12.64', 'train_loss': '0.4259', 'epoch': '15'}


TrainOutput(global_step=3210, training_loss=0.4259488314854393, metrics={'train_runtime': 253.9187, 'train_samples_per_second': 50.39, 'train_steps_per_second': 12.642, 'train_loss': 0.4259488314854393, 'epoch': 15.0})

### C.5 Ocena, zapis adaptera, wczytanie, predykcje

Zastrzeżenie metodologiczne odziedziczone po artykule: **te same** 212 tekstów testowych służy zarówno do wyboru najlepszej epoki (`load_best_model_at_end`), jak i do raportowania trafności, więc wynik jest nieco zawyżony (model "podglądnął" test przy wyborze epoki). W zadaniu 1c wydziel osobny zbiór testowy.

In [62]:
in_memory_acc = (
    predict_ids(med_model, raw["test"]["input_text"]) == test_labels
).mean() * 100
print(
    f"Po douczeniu (w pamięci)  : {in_memory_acc:.2f}%  | bazowy {baseline_acc:.2f}% | artykuł 94,34%"
)

adapter_dir = ARTIFACTS / "bert-lora-symptoms"
med_model.save_pretrained(adapter_dir)
for f in sorted(adapter_dir.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size / 1024:9.1f} KB")

# wczytanie: świeży model bazowy + adapter
fresh = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=len(label2id), label2id=label2id, id2label=id2label
)
reloaded_model = PeftModel.from_pretrained(fresh, adapter_dir)
reloaded_acc = (
    predict_ids(reloaded_model, raw["test"]["input_text"]) == test_labels
).mean() * 100
print(f"Po douczeniu (wczytany)   : {reloaded_acc:.2f}%")

sample = raw["test"].select(range(5))
for text, true, pred in zip(
    sample["input_text"],
    sample["label"],
    predict_ids(reloaded_model, sample["input_text"]),
):
    print(
        f"\n{text[:110]}...\n  prawdziwa: {true:22s} przewidziana: {id2label[int(pred)]}"
    )


Po douczeniu (w pamięci)  : 95.28%  | bazowy 6.60% | artykuł 94,34%
  README.md                          5.0 KB
  adapter_config.json                1.1 KB
  adapter_model.safetensors       9288.9 KB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Po douczeniu (wczytany)   : 95.28%

I have a burning sensation in my stomach that comes and goes. It's worse when I eat and when I lie down. I als...
  prawdziwa: peptic ulcer disease   przewidziana: peptic ulcer disease

I have a hard time swallowing and I feel like food is getting stuck in my throat. I belch a lot and feel bloat...
  prawdziwa: peptic ulcer disease   przewidziana: peptic ulcer disease

I've been having headaches and migraines, and I can't sleep. My whole body shakes and twitches. Sometimes I fe...
  prawdziwa: drug reaction          przewidziana: drug reaction

I'm sweating a lot and can't catch my breath. My throat is full of phlegm and I feel awful. My heart is racing...
  prawdziwa: pneumonia              przewidziana: pneumonia

I've been scratching myself a lot lately, and my skin is covered in rashy places. I also have a few pimples th...
  prawdziwa: fungal infection       przewidziana: fungal infection


### C.6 Refleksja (odpowiedz własnymi słowami)

1. Skok z ok. 5% do ok. 95%: które parametry były uczone i jaką część całego modelu to stanowi?
2. Adapter waży kilka MB. Co jeszcze **musisz** mieć, żeby go użyć?
3. `lora_alpha=1, r=64` daje mnożnik 1/64, a mimo to trening działa przy tempie uczenia `lr=2e-3`. Dlaczego mały mnożnik w połączeniu z dużym tempem uczenia może działać?
4. Co się zmieni w tym procesie przy **klasyfikacji tokenów** (`BertForTokenClassification`)? Pomyśl o: etykietach (A.6), collatorze, metryce, `task_type`.
5. Co się zmieni dla **GPT-2**? Przeczytaj najpierw Część D.

# Część D
## Klasy GPT-2 w bibliotece `transformers`

**W prostych słowach.** GPT-2 to model, który nauczył się jednej rzeczy: **dopisywać następne słowo**. Czyta tekst od lewej do prawej i za każdym razem zgaduje, co będzie dalej. Powtarzając to w kółko, potrafi napisać całe zdania.

Kluczowa różnica względem BERT-a to **maska przyczynowa**: GPT-2 przy przetwarzaniu słowa numer $t$ widzi tylko słowa $1 \dots t$, a przyszłości nie (bo przy generowaniu przyszłość jeszcze nie istnieje). BERT widzi całe zdanie naraz.

| | BERT (koder, *encoder*) | GPT-2 (dekoder, *decoder*) |
|---|---|---|
| Uwaga | dwukierunkowa | **przyczynowa**: token $t$ widzi tylko tokeny $\le t$ |
| Czego był uczony | zgadywania zasłoniętych słów | zgadywania następnego słowa |
| Tokeny specjalne | `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]` | tylko `<\|endoftext\|>` (jako koniec tekstu i, umownie, jako padding) |
| Informacja o pozycji | uczone osadzenia pozycji | uczone osadzenia pozycji (`wpe`) |
| Osadzenia segmentów | tak (`token_type_ids`) | nie |
| Normalizacja (LayerNorm) | po warstwie | przed warstwą + końcowa `ln_f` |
| Pooler | opcjonalny | brak |
| Warstwy liniowe | `nn.Linear` | `Conv1D` (to `Linear` z **transponowaną** macierzą wag) |

**Konsekwencja dla klasyfikacji.** Skoro uwaga jest przyczynowa, tylko **ostatni token** widział cały tekst. Nie ma tu `[CLS]` na początku, który zebrałby sens całego zdania (na początku nie widzi jeszcze niczego poza sobą). Dlatego głowice klasyfikacji GPT-2 czytają **ostatni prawdziwy token** (nie padding).

### D.1 `GPT2Model`: "mózg" GPT-2

Elementy: `wte` (osadzenia tokenów), `wpe` (osadzenia pozycji), `drop` (dropout), `h` (lista bloków, tu 12), `ln_f` (końcowa normalizacja). Każdy blok ma: `ln_1`, `attn` (`c_attn` = jedna warstwa, która naraz liczy Q, K i V, oraz `c_proj`), `ln_2`, `mlp` (`c_fc`, `c_proj`).

Budowa jest bardzo podobna do BERT-a (osadzenia -> 12 bloków uwagi i sieci -> wektory). Różnią się maską uwagi i drobiazgami z tabeli.

In [63]:
gtok = AutoTokenizer.from_pretrained("gpt2")
gtok.pad_token = (
    gtok.eos_token
)  # GPT-2 nie ma tokenu paddingu; standardowe obejście to użycie eos

gpt = GPT2Model.from_pretrained("gpt2").eval()
gcfg = gpt.config
print(
    {
        k: getattr(gcfg, k)
        for k in ["n_embd", "n_layer", "n_head", "n_positions", "vocab_size"]
    },
    "\n",
)
print(gpt.wte, gpt.wpe, sep="\n")
print()
print(gpt.h[0])
print()
print(gpt.ln_f)

g_batch = gtok(sentences, padding=True, return_tensors="pt")
print("\ntokeny:", [gtok.convert_ids_to_tokens(r) for r in g_batch["input_ids"]])
print("attention_mask:", g_batch["attention_mask"].tolist())
print("strona paddingu:", gtok.padding_side)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

{'n_embd': 768, 'n_layer': 12, 'n_head': 12, 'n_positions': 1024, 'vocab_size': 50257} 

Embedding(50257, 768)
Embedding(1024, 768)

GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)

tokeny: [['The', 'Ġpatient', 'Ġhas', 'Ġa', 'Ġhigh', 'Ġfever', '.'], ['Hello', 'Ġworld', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>']]
attention_mask: [[1, 1, 1, 1, 1, 1, 1], [1, 1, 0, 0, 0, 0, 0]]
strona paddingu: right


**Przewidź.** Dla `g_batch` (wypisanego wyżej): jaki kształt ma `last_hidden_state` w `GPT2Model`? I które z dwóch wyjść BERT-a (`last_hidden_state`, `pooler_output`) zwraca `GPT2Model`?

In [ ]:
pred_gpt_hidden = None  # (B, T, H) z wpisanymi liczbami
pred_gpt_keys = None  # krotka nazw wyjść, np. ("last_hidden_state", "pooler_output")


In [65]:
with torch.no_grad():
    g_out = gpt(**g_batch)

check("GPT2Model last_hidden_state", pred_gpt_hidden, g_out.last_hidden_state.shape)
# past_key_values to pamięć podręczna do generowania, a nie reprezentacja: pomijamy ją
actual_keys = tuple(
    k for k in g_out.keys() if g_out[k] is not None and k != "past_key_values"
)
check("GPT2Model klucze wyniku", pred_gpt_keys, actual_keys)


[GPT2Model last_hidden_state] brak prognozy. Prawdziwa wartość: torch.Size([2, 7, 768])
[GPT2Model klucze wyniku] brak prognozy. Prawdziwa wartość: ('last_hidden_state',)


### D.2 Maska przyczynowa w praktyce

Zasada: wektor tokenu na pozycji $t$ zależy tylko od tokenów $\le t$. Sprawdźmy to eksperymentem. Zmieniamy **ostatni** token zdania i patrzymy, czy zmieniły się wektory wcześniejszych tokenów. Robimy to dla GPT-2 i dla BERT-a.

Intuicja: jeśli słowo na końcu zmienimy, a GPT-2 patrzy tylko "w lewo", to wcześniejsze słowa nie mają jak się o tym dowiedzieć.

**Przewidź.** Dla którego modelu wcześniejsze pozycje pozostaną identyczne, a co stanie się z wcześniejszymi pozycjami w BERT-cie?

In [ ]:
def hidden(model, enc):
    with torch.no_grad():
        return model(**enc).last_hidden_state[0]


# GPT-2: podmieniamy ostatni token
a = gtok("The cat sat on the mat", return_tensors="pt")
b = {k: v.clone() for k, v in a.items()}
b["input_ids"][0, -1] = gtok.encode(" rug")[0]
ha, hb = hidden(gpt, a), hidden(gpt, b)
print(
    "GPT-2 wcześniejsze pozycje bez zmian:",
    torch.allclose(ha[:-1], hb[:-1], atol=1e-5),
    "| ostatnia pozycja się zmieniła:",
    not torch.allclose(ha[-1], hb[-1], atol=1e-5),
)

# BERT: podmieniamy ostatnie prawdziwe słowo (token przed [SEP])
a = tok("The cat sat on the mat", return_tensors="pt")
b = {k: v.clone() for k, v in a.items()}
b["input_ids"][0, -2] = tok.convert_tokens_to_ids("rug")
ha, hb = hidden(bert, a), hidden(bert, b)
print(
    "BERT  wcześniejsze pozycje bez zmian:", torch.allclose(ha[:-2], hb[:-2], atol=1e-5)
)
print(
    "BERT  największa zmiana we wcześniejszej pozycji:",
    (ha[:-2] - hb[:-2]).abs().max().item(),
)


GPT-2 wcześniejsze pozycje bez zmian: True | ostatnia pozycja się zmieniła: True
BERT  wcześniejsze pozycje bez zmian: False
BERT  największa zmiana we wcześniejszej pozycji: 1.5463602542877197


### D.3 `GPT2LMHeadModel`: dopisywanie następnego słowa

**W prostych słowach.** To "pełny" GPT-2 do pisania tekstu. Po przeczytaniu początku zdania model podaje, które słowo ze słownika jest najbardziej prawdopodobne jako następne.

Budowa: `GPT2Model` + głowica `lm_head = Linear(H, V, bias=False)`, której wagi są **te same** co osadzenia wejściowe `wte` (jak w BERT-owym MLM). Wynik: `(B, T, V)`, gdzie `logits[:, t]` ocenia, jakie słowo stoi na pozycji $t+1$.

**Strata.** Model sam przesuwa etykiety o jedno miejsce. Wystarczy podać `labels=input_ids`, a obliczy on entropię krzyżową między `logits[:, :-1]` a `labels[:, 1:]`. (Pozycja $t$ ma przewidzieć token $t+1$.) Miejsca paddingu w `labels` powinny mieć `-100`.

**Generowanie.** `generate()` (zadanie 5) działa w pętli:
1. bierze `logits[:, -1]` (ocena następnego słowa),
2. wybiera słowo: *zachłannie* (zawsze najlepsze) albo *losowo* według prawdopodobieństw, sterowane parametrami: `temperature` (im wyższa, tym bardziej "szalone" wybory), `top_k` (losuj tylko spośród $k$ najlepszych), `top_p` (losuj spośród najlepszych, których prawdopodobieństwa sumują się do $p$),
3. dopisuje słowo do tekstu i wraca do punktu 1.

In [67]:
lm = GPT2LMHeadModel.from_pretrained("gpt2").eval()
print(lm.lm_head)
print(
    "lm_head ma te same wagi co wte:",
    lm.lm_head.weight.data_ptr() == lm.transformer.wte.weight.data_ptr(),
)

enc = gtok("The quick brown fox jumps over the lazy dog", return_tensors="pt")
with torch.no_grad():
    out_lm = lm(**enc, labels=enc["input_ids"])
print("\nlogity:", tuple(out_lm.logits.shape), "= (B, T, V)")

manual = F.cross_entropy(out_lm.logits[0, :-1], enc["input_ids"][0, 1:])
print(
    "strata == CE(logity[:-1], ids[1:]):",
    torch.allclose(manual, out_lm.loss, atol=1e-5),
    "| strata =",
    round(out_lm.loss.item(), 3),
)

prompt = gtok("The capital of France is", return_tensors="pt")
with torch.no_grad():
    probs = lm(**prompt).logits[0, -1].softmax(-1)
print("\n5 najbardziej prawdopodobnych następnych tokenów:")
for p, i in zip(*probs.topk(5)):
    print(f"  {gtok.decode(i)!r:12} {p.item():.3f}")

gen = lm.generate(
    **prompt, max_new_tokens=12, do_sample=False, pad_token_id=gtok.eos_token_id
)
print("\nzachłannie:", gtok.decode(gen[0]))
torch.manual_seed(0)
gen = lm.generate(
    **prompt,
    max_new_tokens=12,
    do_sample=True,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    pad_token_id=gtok.eos_token_id,
)
print("losowo:", gtok.decode(gen[0]))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Linear(in_features=768, out_features=50257, bias=False)
lm_head ma te same wagi co wte: True

logity: (1, 9, 50257) = (B, T, V)
strata == CE(logity[:-1], ids[1:]): True | strata = 5.426

5 najbardziej prawdopodobnych następnych tokenów:
  ' the'       0.085
  ' now'       0.048
  ' a'         0.046
  ' France'    0.032
  ' Paris'     0.032

zachłannie: The capital of France is the capital of the French Republic, and the capital of the
losowo: The capital of France is a place that has never had much of a French culture,


### D.4 `GPT2ForSequenceClassification`: klasyfikacja po ostatnim tokenie

**W prostych słowach.** Tak samo jak w BERT-cie: jedno zdanie -> jedna etykieta. Ale GPT-2 nie ma `[CLS]` na początku. Zamiast tego bierzemy wektor **ostatniego prawdziwego tokenu**, bo tylko on "przeczytał" całe zdanie (D.2).

Budowa: `GPT2Model` -> `score = Linear(H, L, bias=False)` stosowane do **każdej** pozycji, a następnie model wybiera wynik z pozycji **ostatniego tokenu niebędącego paddingiem** w każdej sekwencji.

Pułapki:
- GPT-2 nie ma tokenu paddingu. Przy paczkach większych niż 1 model musi znać `pad_token_id`, żeby znaleźć ostatni prawdziwy token (ustaw `tokenizer.pad_token = tokenizer.eos_token` i `model.config.pad_token_id`).
- Skoro padding == eos, to prawdziwy `<|endoftext|>` w tekście jest nieodróżnialny od paddingu, jeśli indeks liczymy z `input_ids`. Pamiętaj o tym, gdy dopisujesz eos do przykładów.
- To właśnie jest polecenie z zadania 3: "ostatni token zamiast `[CLS]`".

In [68]:
g_seq = GPT2ForSequenceClassification.from_pretrained(
    "gpt2", num_labels=3, pad_token_id=gtok.pad_token_id
).eval()
print(g_seq.score, "\n")

with torch.no_grad():
    logits_seq = g_seq(**g_batch).logits
    h_all = g_seq.transformer(**g_batch).last_hidden_state

last_idx = (
    g_batch["attention_mask"].sum(1) - 1
)  # padding z prawej: indeks ostatniego prawdziwego tokenu
print(
    "indeks ostatniego prawdziwego tokenu w każdej sekwencji:",
    last_idx.tolist(),
    "przy T =",
    g_batch["input_ids"].shape[1],
)
manual_logits = g_seq.score(h_all[torch.arange(2), last_idx])
print(
    "logity == score(h[ostatni prawdziwy token]):",
    torch.allclose(manual_logits, logits_seq, atol=1e-5),
)

# co się stanie bez pad_token_id przy paczce > 1?
no_pad = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=3).eval()
try:
    with torch.no_grad():
        no_pad(**g_batch)
    print("brak błędu bez pad_token_id")
except Exception as e:
    print(f"\nbez pad_token_id -> {type(e).__name__}: {e}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Linear(in_features=768, out_features=3, bias=False) 

indeks ostatniego prawdziwego tokenu w każdej sekwencji: [6, 1] przy T = 7
logity == score(h[ostatni prawdziwy token]): True


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


bez pad_token_id -> ValueError: Cannot handle batch sizes > 1 if no padding token is defined.


### D.5 `GPT2ForTokenClassification`: etykieta dla każdego tokenu

**W prostych słowach.** Jak `BertForTokenClassification`: każde słowo dostaje własną etykietę. Różnica: GPT-2 jest przyczynowy, więc etykietę tokenu wyznacza **tylko to, co jest przed nim** (kontekst z lewej).

Budowa: `GPT2Model` -> `dropout` -> `classifier = Linear(H, L)` na każdej pozycji. Wynik: `(B, T, L)`.

Zgodnie z zadaniem 3, tokeny paddingu i znacznik `<eos>` muszą dostać etykietę `-100`, żeby nie liczyły się do straty. Sprawdźmy to, licząc stratę ręcznie.

In [69]:
g_tok = GPT2ForTokenClassification.from_pretrained("gpt2", num_labels=5).eval()
print(g_tok.dropout, g_tok.classifier, "\n")

torch.manual_seed(0)
labels = torch.randint(0, 5, g_batch["input_ids"].shape)
labels[g_batch["attention_mask"] == 0] = -100  # padding
# eos (tutaj to to samo co padding; po dopisaniu <eos> do tekstu byłaby to osobna pozycja)
labels[g_batch["input_ids"] == gtok.eos_token_id] = -100
print(labels)

with torch.no_grad():
    out_gt = g_tok(**g_batch, labels=labels)
print("\nlogity:", tuple(out_gt.logits.shape), "= (B, T, L)")
manual = nn.CrossEntropyLoss(ignore_index=-100)(
    out_gt.logits.view(-1, 5), labels.view(-1)
)
print(
    "strata == CrossEntropyLoss(ignore_index=-100):",
    torch.allclose(manual, out_gt.loss, atol=1e-5),
)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Dropout(p=0.1, inplace=False) Linear(in_features=768, out_features=5, bias=True) 

tensor([[   4,    4,    3,    0,    3,    4,    2],
        [   3,    2, -100, -100, -100, -100, -100]])

logity: (2, 7, 5) = (B, T, L)
strata == CrossEntropyLoss(ignore_index=-100): True


### D.6 PEFT na GPT-2: co jest inaczej

Dwie różnice względem BERT-a, obie widoczne na wydruku `gpt.h[0]` wyżej:

1. **Nazwy warstw docelowych.** BERT ma osobne warstwy `query`/`key`/`value`. GPT-2 skleja je w jedną `c_attn`, więc `target_modules=["c_attn"]` adaptuje Q, K i V naraz.
2. **Warstwy `Conv1D`** trzymają wagi jako `(wejście, wyjście)`, czyli transponowane względem `nn.Linear`. PEFT potrzebuje `fan_in_fan_out=True`, żeby nałożyć poprawkę LoRA w dobrej orientacji (PEFT ostrzeże i ustawi to sam, jeśli zapomnisz, ale jawnie jest czytelniej).

**Przewidź.** Przy `r=8` tylko na `c_attn` (`768 -> 2304`) w 12 blokach: ile parametrów LoRA? Pamiętaj, że para LoRA kosztuje $r (d_{in} + d_{out})$.

In [70]:
pred_gpt_lora = None  # jedna liczba całkowita


In [71]:
g_base = GPT2ForSequenceClassification.from_pretrained(
    "gpt2", num_labels=3, pad_token_id=gtok.pad_token_id
)
g_lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],
    fan_in_fan_out=True,
)
g_peft = get_peft_model(g_base, g_lora_cfg)
g_peft.print_trainable_parameters()

names = [n for n, p in g_peft.named_parameters() if p.requires_grad]
print(
    "uczone, ale to nie LoRA:", [n for n in names if "lora_" not in n]
)  # głowica zadania: 'score'
actual = sum(
    p.numel() for n, p in g_peft.named_parameters() if p.requires_grad and "lora_" in n
)
check(
    "parametry LoRA w GPT-2",
    None if pred_gpt_lora is None else (pred_gpt_lora,),
    (actual,),
)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

trainable params: 297,216 || all params: 124,739,328 || trainable%: 0.2383
uczone, ale to nie LoRA: ['base_model.model.score.modules_to_save.default.weight']
[parametry LoRA w GPT-2] brak prognozy. Prawdziwa wartość: (294912,)


**Twoja kolej (D).** Zadanie 3 każe użyć reprezentacji ostatniego tokenu jako osadzenia tekstu. Zaimplementuj `last_token_embedding(last_hidden_state, attention_mask)`, która zwraca `(B, H)`: wektor z pozycji **ostatniego prawdziwego** tokenu każdej sekwencji. Załóż padding **z prawej** (jak w D.4). Potem zastanów się: co się zmieni, gdy tokenizer dopycha **z lewej**?

In [72]:
def last_token_embedding(last_hidden_state, attention_mask):
    # TODO: (B, T, H) i (B, T) -> (B, H)
    return None


In [73]:
expected = torch.stack(
    [g_out.last_hidden_state[0, 6], g_out.last_hidden_state[1, 1]]
)  # ostatnie prawdziwe tokeny, z D.4
mine = last_token_embedding(g_out.last_hidden_state, g_batch["attention_mask"])
print(
    "jeszcze niezaimplementowane"
    if mine is None
    else f"zgodne z referencją: {torch.allclose(mine, expected)}"
)

# padding z lewej: gdzie teraz jest ostatni prawdziwy token?
left_tok = AutoTokenizer.from_pretrained("gpt2", padding_side="left")
left_tok.pad_token = left_tok.eos_token
left = left_tok(sentences, padding=True, return_tensors="pt")
print("\nattention_mask przy paddingu z lewej:", left["attention_mask"].tolist())


jeszcze niezaimplementowane

attention_mask przy paddingu z lewej: [[1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 1, 1]]


# Podsumowanie i samosprawdzenie

| | BERT | GPT-2 |
|---|---|---|
| Ciało ("mózg") | `BertModel` | `GPT2Model` |
| Głowica językowa | `BertForMaskedLM`: dla każdego tokenu wynik dla każdego słowa ze słownika; strata tylko na zasłoniętych miejscach | `GPT2LMHeadModel`: dla każdego tokenu ocena następnego słowa; etykiety przesuwane automatycznie |
| Klasyfikacja tekstu | `BertForSequenceClassification`: `pooler_output` z `[CLS]` -> `Linear` | `GPT2ForSequenceClassification`: **ostatni prawdziwy token** -> `Linear` (wymaga `pad_token_id`) |
| Klasyfikacja tokenów | `BertForTokenClassification`: każdy token -> `Linear` (kontekst z obu stron) | `GPT2ForTokenClassification`: każdy token -> `Linear` (tylko kontekst z lewej) |
| `task_type` w PEFT | `SEQ_CLS` / `TOKEN_CLS` | `SEQ_CLS` / `TOKEN_CLS`, a dla głowicy językowej `CAUSAL_LM` |
| Gdzie LoRA | `query`, `value` (także `key`, `dense`) | `c_attn` z `fan_in_fan_out=True` |

Rozumiesz Część I, jeśli potrafisz odpowiedzieć **bez przewijania w górę**:

1. Czym różni się `last_hidden_state` od `pooler_output` i które głowice z czego korzystają?
2. Po co jest etykieta `-100` i skąd biorą się pozycje `-100` w klasyfikacji tokenów?
3. W LoRA: dlaczego `B` startuje od zera, a `A` jest losowe? Co by się zepsuło, gdyby obie macierze były zerowe?
4. Które parametry są uczone po `get_peft_model(..., task_type=SEQ_CLS)` i dlaczego jest wśród nich głowica klasyfikacji?
5. Dlaczego klasyfikacja w GPT-2 czyta *ostatni* token, a w BERT-cie *pierwszy*?
6. Co się psuje przy paczkach w GPT-2, gdy nie ustawisz `pad_token_id`?

**Dalej (poza tym notatnikiem):** zadania 1-5 z laboratorium stosują dokładnie te elementy do polskich modeli (`allegro/herbert-base-cased`, `sdadas/polish-gpt2-medium`) i danych z Laboratorium 1: douczanie z PEFT do klasyfikacji tekstu i tokenów, wizualizacja osadzeń z `[CLS]` / ostatniego tokenu oraz augmentacja danych z użyciem BERT-a i GPT-2.